In [1]:
import copy
import joblib
import functools
from tqdm import tqdm
import torch.optim as optim
from torchvision import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
from torch.autograd import Variable
import itertools
import operator
from itertools import islice
from collections import OrderedDict

#SimpleDMO opt class

In [2]:
#with_bias
class SDMOptimizer(nn.Module):
    def __init__(self, preproc=False, hidden_sz=20, preproc_factor=10.0):
        super().__init__()
        self.hidden_sz = hidden_sz
        if preproc:
            self.recurs = nn.LSTMCell(2, hidden_sz) #took input as sign and pre-processes grad
        else:
            self.recurs = nn.LSTMCell(1, hidden_sz)

        self.recurs2 = nn.LSTMCell(hidden_sz, hidden_sz)
        self.output = nn.Linear(2*hidden_sz, 1)
        self.preproc = preproc
        self.preproc_factor = preproc_factor
        self.preproc_threshold = np.exp(-preproc_factor)
        #some real numbers multiplied to hidden states as parameters
        self.alpha = nn.Parameter(torch.Tensor(1))
        self.beta = nn.Parameter(torch.Tensor(1))

    def forward(self, inp, hidden, cell):
        if self.preproc:
            # Implement preproc described in Appendix A

            # Note: we do all this work on tensors, which means
            # the gradients won't propagate through inp. This
            # should be ok because the algorithm involves
            # making sure that inp is already detached.
            inp = inp.data
            inp2 = w(torch.zeros(inp.size()[0], 2))
            keep_grads = (torch.abs(inp) >= self.preproc_threshold).squeeze()
            inp2[:, 0][keep_grads] = (torch.log(torch.abs(inp[keep_grads]) + 1e-8) / self.preproc_factor).squeeze()
            inp2[:, 1][keep_grads] = torch.sign(inp[keep_grads]).squeeze()

            inp2[:, 0][~keep_grads] = -1
            inp2[:, 1][~keep_grads] = (float(np.exp(self.preproc_factor)) * inp[~keep_grads]).squeeze()
            inp = w(Variable(inp2))

        #LSTMCell takes in input (hidden, cell) and outputs the same although the initialization doesnt say it
        #eg. hx, cx = rnn(input[i], (hx, cx))
        #can take inputs in batches, hidden state in batches and cell state in batches
        hidden0, cell0 = self.recurs(inp, (hidden[0], cell[0]))
        hidden1, cell1 = self.recurs2(hidden0, (hidden[1], cell[1]))

        sum_hidden_states = torch.zeros_like(hidden1[0])
        n_h = len(hidden1)
        for i in range(n_h):
            sum_hidden_states = sum_hidden_states + hidden1[i]

        average_hidden = (1/n_h)*sum_hidden_states

        return self.output(torch.cat((self.alpha*hidden1 , self.beta*torch.tile(average_hidden, (n_h, 1))), dim=1)),\
                (hidden0, hidden1), (cell0, cell1)



#DMO


In [3]:
class Optimizer(nn.Module):
    def __init__(self, preproc=False, hidden_sz=20, preproc_factor=10.0):
        super().__init__()
        self.hidden_sz = hidden_sz
        if preproc:
            self.recurs = nn.LSTMCell(2, hidden_sz)
        else:
            self.recurs = nn.LSTMCell(1, hidden_sz)
        self.recurs2 = nn.LSTMCell(hidden_sz, hidden_sz)
        self.output = nn.Linear(hidden_sz, 1)
        self.preproc = preproc
        self.preproc_factor = preproc_factor
        self.preproc_threshold = np.exp(-preproc_factor)

    def forward(self, inp, hidden, cell):
        if self.preproc:
            # Implement preproc described in Appendix A

            # Note: we do all this work on tensors, which means
            # the gradients won't propagate through inp. This
            # should be ok because the algorithm involves
            # making sure that inp is already detached.
            ################################################################################################################################

            #   LOOK AT THE L2LGDGD Paper for the "gradient preprocessing" in Appendix

            ##############################################################################################################
            #the gradients

            inp = inp.data

            inp2 = w(torch.zeros(inp.size()[0], 2))

            #boolean to see which gradients pass the threshold (gradient cap)
            keep_grads = (torch.abs(inp) >= self.preproc_threshold).squeeze()
            inp2[:, 0][keep_grads] = (torch.log(torch.abs(inp[keep_grads]) + 1e-8) / self.preproc_factor).squeeze()

            #keeping all the signs of the gradients that have "true" value in keep_grads
            inp2[:, 1][keep_grads] = torch.sign(inp[keep_grads]).squeeze()

            inp2[:, 0][~keep_grads] = -1
            inp2[:, 1][~keep_grads] = (float(np.exp(self.preproc_factor)) * inp[~keep_grads]).squeeze()
            inp = w(Variable(inp2))
        hidden0, cell0 = self.recurs(inp, (hidden[0], cell[0]))
        hidden1, cell1 = self.recurs2(hidden0, (hidden[1], cell[1]))
        return self.output(hidden1), (hidden0, hidden1), (cell0, cell1)

#Resnet Model class

##meta_module

In [4]:


def to_var(x, requires_grad=True):
    if torch.cuda.is_available():
        x = x.cuda()
    return Variable(x, requires_grad=requires_grad)


class MetaModule(nn.Module):
    # adopted from: Adrien Ecoffet https://github.com/AdrienLE
    def parameters(self):
       for name, param in self.named_params(self):
            yield param

    def named_parameters(self):
       for name, param in self.named_params(self):
            yield name, param

    def named_leaves(self):
        return []

    def named_submodules(self):
        return []

    def named_params(self, curr_module=None, memo=None, prefix=''):
        if memo is None:
            memo = set()

        if hasattr(curr_module, 'named_leaves'):
            for name, p in curr_module.named_leaves():
                if p is not None and p not in memo:
                    memo.add(p)
                    yield prefix + ('.' if prefix else '') + name, p
        else:
            for name, p in curr_module._parameters.items():
                if p is not None and p not in memo:
                    memo.add(p)
                    yield prefix + ('.' if prefix else '') + name, p

        for mname, module in curr_module.named_children():
            submodule_prefix = prefix + ('.' if prefix else '') + mname
            for name, p in self.named_params(module, memo, submodule_prefix):
                yield name, p

    def update_params(self, lr_inner, first_order=False, source_params=None, detach=False):
        if source_params is not None:
            for tgt, src in zip(self.named_params(self), source_params):
                name_t, param_t = tgt
                # name_s, param_s = src
                # grad = param_s.grad
                # name_s, param_s = src
                grad = src
                if first_order:
                    grad = to_var(grad.detach().data)
                tmp = param_t - lr_inner * grad
                self.set_param(self, name_t, tmp)
        else:
            for name, param in self.named_params(self):
                if not detach:
                    grad = param.grad
                    if first_order:
                        grad = to_var(grad.detach().data)
                    tmp = param - lr_inner * grad
                    self.set_param(self, name, tmp)
                else:
                    param = param.detach_()
                    self.set_param(self, name, param)

    def set_param(self,curr_mod, name, param):
        if '.' in name:
            n = name.split('.')
            module_name = n[0]
            rest = '.'.join(n[1:])
            for name, mod in curr_mod.named_children():
                if module_name == name:
                    self.set_param(mod, rest, param)
                    break
        else:
            setattr(curr_mod, name, param)

    def detach_params(self):
        for name, param in self.named_params(self):
            self.set_param(self, name, param.detach())

    def copy(self, other, same_var=False):
        for name, param in other.named_params():
            if not same_var:
                param = to_var(param.data.clone(), requires_grad=True)
            self.set_param(name, param)





##resnet

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class ResNet_Model(MetaModule):
    def __init__(self, block, layers, num_classes = 10):
        super(ResNet_Model, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)
        self.avgpool = nn.AvgPool2d(7, stride=1, padding = 3)
        self.fc = nn.Linear(512, num_classes)
        self.loss = nn.CrossEntropyLoss()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:

            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)


    def forward(self, data_batch):

        x, y = data_batch.sample()
        x = w(Variable(x))
        y = w(Variable(y))
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)  #no need of softmax layer since it is already in torch.nn.CrossEntropyLoss
        loss = self.loss(x, y)
        return loss, x, y

In [6]:
USE_CUDA = torch.cuda.is_available()

def w(v):
    if USE_CUDA:
        return v.cuda()
    return v

In [7]:
USE_CUDA

True

#data set and loss

In [8]:
from google.colab import drive
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [9]:

class Cifar100_Data:
    def __init__(self, batch_size=128):
        dataset_train = datasets.CIFAR10(
            "/content/drive/MyDrive/Mtech Project/codes/chen2020/", train=True, download=True,
            transform=torchvision.transforms.Compose([ torchvision.transforms.ToTensor()]) #torchvision.transforms.Resize(32),
        )

        self.batch_size= batch_size
        self.loader = torch.utils.data.DataLoader(
            dataset_train, batch_size=self.batch_size, num_workers=32)

        self.batches = []
        self.cur_batch = 0

    def sample(self):
        if self.cur_batch >= len(self.batches):
            self.batches = []
            self.cur_batch = 0
            for b in self.loader:
                self.batches.append(b)
        batch = self.batches[self.cur_batch]
        self.cur_batch += 1
        return batch


# for validation

dataset_test = datasets.CIFAR10(
            "/content/drive/MyDrive/Mtech Project/codes/chen2020/", train=False, download=True,
            transform=torchvision.transforms.Compose([ torchvision.transforms.ToTensor()])) #torchvision.transforms.Resize(32),

val_data_loader = torch.utils.data.DataLoader(
            dataset_test, batch_size=16, num_workers=32)



100%|██████████| 170498071/170498071 [00:13<00:00, 12193636.79it/s]


Extracting /content/drive/MyDrive/Mtech Project/codes/chen2020/cifar-10-python.tar.gz to /content/drive/MyDrive/Mtech Project/codes/chen2020/


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [10]:
dataset_test[0][0].shape

torch.Size([3, 32, 32])

In [11]:
criterion = nn.CrossEntropyLoss()
def accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    acc = correct / total
    return acc

# pretrained LO


In [12]:
#loading pretrained LO model (DMO)
DMO_dict = torch.load("/content/drive/MyDrive/Mtech Project/codes/chen2020/CPpoint_LO_lr_0.003.pth")

DMO_instance = w(Optimizer(preproc=True))
DMO_instance.load_state_dict(DMO_dict['optimizer_state_dict'])

<All keys matched successfully>

In [13]:
SDMO_dict = torch.load("/content/drive/MyDrive/Mtech Project/codes/chen2020/CPpoint_sDMO_lr_0.003.pth")

SDMO_instance = w(SDMOptimizer(preproc=True))
SDMO_instance.load_state_dict(SDMO_dict['optimizer_state_dict'])

<All keys matched successfully>

#training function

In [14]:
# training function for Resnet

def detach_var(v):
    var = w(Variable(v.data, requires_grad=True))
    var.retain_grad()
    return var

def rgetattr(obj, attr, *args):
    def _getattr(obj, attr):
        return getattr(obj, attr, *args)
    return functools.reduce(_getattr, [obj] + attr.split('.'))


def rsetattr(obj, attr, val):
    pre, _, post = attr.rpartition('.')
    return setattr(rgetattr(obj, pre) if pre else obj, post, val)




def fit_val_loss_acc(opt_net, target_data, target_optimizee, optim_it, out_mul=1):

    #use the should_train parameter to evaluate the task and not train the

    opt_net.eval()   #switch for some specific layers/parts of the model that behave differently during training and inference (evaluating) time. For example, Dropouts Layers, BatchNorm Layers etc.

    target = target_data  #it is the dataset, optimizee(target) yields loss, output
    optimizee = w(target_optimizee)
    n_params = 0
    for name, p in optimizee.named_parameters():
        n_params += int(np.prod(p.size()))

    #there are two outputs of LSTM (long term= Cell State C_t, and Hidden State = h_t)
    hidden_states = [w(Variable(torch.zeros(n_params, opt_net.hidden_sz))) for _ in range(2)]
    cell_states = [w(Variable(torch.zeros(n_params, opt_net.hidden_sz))) for _ in range(2)]



    ###############  keep track of the gradient values at the inference time  ########################

    # grad_list_test = []
    ##############################

    train_loss = []
    test_loss= []
    train_acc = []
    test_acc = []

    for iteration in tqdm(range(1, optim_it + 1), "iteration: "):  #optim_it = number of iterations the optimizee is trained using LO(inner training steps)

        #the loss is always found using all the datapoints (not single or batch loss)
        # difference between consecutive losses will decrease quadratically


        loss, pred, real  = optimizee(target)
        train_loss_iter = loss.item()
        train_loss.append(train_loss_iter)
        tr_acc = accuracy(pred, real)
        train_acc.append(tr_acc)


        # iteration_mod_unroll = iteration % unroll

        loss.backward(retain_graph=False)

        #gradients wrt all the parameters are updated


        offset = 0
        result_params = {}
        hidden_states2 = [w(Variable(torch.zeros(n_params, opt_net.hidden_sz))) for _ in range(2)]
        cell_states2 = [w(Variable(torch.zeros(n_params, opt_net.hidden_sz))) for _ in range(2)]

        grad_list_each_step = []



        for name, p in optimizee.named_parameters():
            cur_sz = int(np.prod(p.size()))
            # We do this so the gradients are disconnected from the graph but we still get
            # gradients from the rest
            gradients = detach_var(p.grad.view(cur_sz, 1))


            ###############  keep track of the gradient values at the inference time  ########################

            grad_list_each_step.append(gradients.tolist())

            ############################################


            updates, new_hidden, new_cell = opt_net(
                gradients,
                [h[offset:offset+cur_sz] for h in hidden_states],
                [c[offset:offset+cur_sz] for c in cell_states]
            )
            #len(new_hidden) = 2

            for i in range(len(new_hidden)):
                hidden_states2[i][offset:offset+cur_sz] = new_hidden[i]
                cell_states2[i][offset:offset+cur_sz] = new_cell[i]
            result_params[name] = p + updates.view(*p.size()) * out_mul
            result_params[name].retain_grad()

            offset += cur_sz


        #####finding the average of hidden states after each iteration of optim_it


        # previous_loss = loss

#######   unroll when necessary to update the params of the LO   ############
        # if iteration_mod_unroll == 0:


            #need to update the grads for optimizee parameters too since they also are connected to the RNN in the computational graph
            #reducing them to zero is necessary
        optimizee = w(target_optimizee)
        optimizee.load_state_dict(result_params)
        optimizee.zero_grad()
        hidden_states = [detach_var(v) for v in hidden_states2]
        cell_states = [detach_var(v) for v in cell_states2]


#######  steps when no unrolling happens ##########################

        # for name, p in optimizee.all_named_parameters():
        #     rsetattr(optimizee, name, result_params[name])
        # assert len(list(optimizee.all_named_parameters()))
        # hidden_states = hidden_states2
        # cell_states = cell_states2


################################################################################

########       VALIDATION: ##########
        loss_val_iter = 0
        pred_y = []
        act_y = []
        for batch in val_data_loader:
            val_x, val_y = batch

            loss_1, valout_pred, val_y = optimizee(batch)
            loss_val_iter += loss_1

            pred_y.extend(valout_pred)
            act_y.extend(val_y)

        acc_val_iter = accuracy(pred_y,act_y)
        test_loss.append(loss_val_iter)
        test_acc.append(acc_val_iter)


        print(f"Training loss:{train_loss_iter} Training Accuracy: {tr_acc} Validation Loss: {loss_val_iter} Validation Accuracy: {acc_val_iter}")
################################################################################


    #all_losses_ever is the loss that optimizee gives after each iteration
    return {"optimizee": optimizee, "train_loss" :train_loss, "train_acc": train_acc, "test_loss": test_loss, "test_acc": test_acc }

#training

In [15]:
history = fit_val_loss_acc(DMO_instance, Cifar100_Data(batch_size=16), ResNet_Model(ResidualBlock, [3, 4, 6, 3], 10), 10)

Files already downloaded and verified


iteration:   0%|          | 0/10 [00:22<?, ?it/s]


OutOfMemoryError: ignored